In [3]:
!pip install rasterstats

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.7/75.7 kB 991.8 kB/s eta 0:00:000:00:01


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from scipy.spatial import cKDTree

# =========================
# INPUTS
# =========================
storm_surge_nc = "/Users/user/Downloads/mocha_storm_surge.nc"
other_hazard_nc = "/Users/user/Downloads/mocha_app_vars.nc"
union_shp = "/Users/user/Downloads/union_shapefile/BD_Union_BBS21.shp"
output_csv = "output_union_damage.csv"

UNION_NAME_FIELD = None  # e.g. "Union"

# Fixed variable logic (as in your successful run)
SURGE_VAR = "z"
U10_VAR = "u10"
V10_VAR = "v10"
RAINNC_VAR = "rainnc"
RAINC_VAR = "rainc"

# =========================
# Helpers
# =========================
LON_CANDS = [
    "lon", "longitude", "xlon", "xlong", "nav_lon",
    "schism_hgrid_node_x", "node_x", "mesh2d_node_x", "x"
]
LAT_CANDS = [
    "lat", "latitude", "ylat", "xlat", "nav_lat",
    "schism_hgrid_node_y", "node_y", "mesh2d_node_y", "y"
]

def _clean_unit(u, fallback="unit"):
    if u is None:
        return fallback
    s = str(u).strip()
    if s == "":
        return fallback
    return s

def _find_coord_var(ds, da, candidates):
    names = list(ds.coords) + list(ds.data_vars)
    lower_map = {n.lower(): n for n in names}

    # exact
    for c in candidates:
        if c.lower() in lower_map:
            nm = lower_map[c.lower()]
            v = ds[nm]
            if len(set(v.dims).intersection(set(da.dims))) > 0 or len(v.dims) == 0:
                return nm

    # contains
    for nm in names:
        nml = nm.lower()
        if any(c.lower() in nml for c in candidates):
            v = ds[nm]
            if len(set(v.dims).intersection(set(da.dims))) > 0 or len(v.dims) == 0:
                return nm

    return None

def _reduce_to_spatial(ds, da, agg="max"):
    lon_name = _find_coord_var(ds, da, LON_CANDS)
    lat_name = _find_coord_var(ds, da, LAT_CANDS)
    if lon_name is None or lat_name is None:
        raise ValueError(
            f"Could not find lon/lat for '{da.name}'. "
            f"da dims={da.dims}, ds vars={list(ds.data_vars)}, coords={list(ds.coords)}"
        )

    lon = ds[lon_name]
    lat = ds[lat_name]

    spatial_dims = set(lon.dims).union(set(lat.dims))
    reduce_dims = [d for d in da.dims if d not in spatial_dims]

    if len(reduce_dims) > 0:
        if agg == "sum":
            da2 = da.sum(dim=reduce_dims, skipna=True)
        else:
            da2 = da.max(dim=reduce_dims, skipna=True)
    else:
        da2 = da

    return da2, lon, lat

def _da_to_points(ds, da, agg="max"):
    da2, lon, lat = _reduce_to_spatial(ds, da, agg=agg)
    lon_b, lat_b, val_b = xr.broadcast(lon, lat, da2)

    x = lon_b.values.ravel().astype(float)
    y = lat_b.values.ravel().astype(float)
    v = val_b.values.ravel().astype(float)

    m = np.isfinite(x) & np.isfinite(y) & np.isfinite(v)
    x, y, v = x[m], y[m], v[m]

    # remove extreme fill values
    m2 = np.abs(v) < 1e20
    x, y, v = x[m2], y[m2], v[m2]

    # 0..360 to -180..180 fix
    if np.nanmin(x) >= 0 and np.nanmax(x) > 180:
        x = np.where(x > 180, x - 360, x)

    pts = gpd.GeoDataFrame(
        {"value": v},
        geometry=gpd.points_from_xy(x, y),
        crs="EPSG:4326"
    )
    return pts

def zonal_mean_from_points(points_gdf, poly_gdf):
    joined = gpd.sjoin(
        points_gdf[["value", "geometry"]],
        poly_gdf[["geometry"]],
        how="left",
        predicate="intersects"
    )

    means = joined.groupby("index_right")["value"].mean()
    out = np.full(len(poly_gdf), np.nan, dtype=float)

    for idx, val in means.items():
        if pd.notna(idx):
            out[int(idx)] = float(val)

    # nearest-point fallback for polygons with no intersecting cells
    miss = np.where(~np.isfinite(out))[0]
    if len(miss) > 0:
        xy = np.column_stack([points_gdf.geometry.x.values, points_gdf.geometry.y.values])
        vv = points_gdf["value"].values
        tree = cKDTree(xy)

        cents = poly_gdf.geometry.centroid
        q = np.column_stack([cents.x.values, cents.y.values])

        k = min(9, len(vv))
        d, ix = tree.query(q[miss], k=k)

        if k == 1:
            out[miss] = vv[ix]
        else:
            out[miss] = np.nanmean(vv[ix], axis=1)

    return out

# =========================
# Load files
# =========================
gdf = gpd.read_file(union_shp)
if gdf.crs is None:
    raise ValueError("Union shapefile CRS is missing.")
    
gdf = gdf.to_crs("EPSG:4326").copy()
gdf = gdf[gdf.geometry.notnull()].copy()
gdf["geometry"] = gdf.geometry.buffer(0)

if UNION_NAME_FIELD and UNION_NAME_FIELD in gdf.columns:
    union_col = UNION_NAME_FIELD
else:
    union_col = next(
        (c for c in ["Union", "union", "UNION", "Name", "name", "UNION_NAME"] if c in gdf.columns),
        gdf.columns[0]
    )

ds_surge = xr.open_dataset(storm_surge_nc)
ds_other = xr.open_dataset(other_hazard_nc)

print("storm_surge_nc vars:", list(ds_surge.data_vars))
print("other_hazard_nc vars:", list(ds_other.data_vars))

# =========================
# Build analysis layers
# =========================
if SURGE_VAR not in ds_surge.data_vars:
    raise ValueError(f"'{SURGE_VAR}' not found in storm_surge_nc.")
da_surge = ds_surge[SURGE_VAR]

if U10_VAR not in ds_other.data_vars or V10_VAR not in ds_other.data_vars:
    raise ValueError(f"'{U10_VAR}'/'{V10_VAR}' not found in other_hazard_nc.")
da_speed = np.hypot(ds_other[U10_VAR], ds_other[V10_VAR]).rename("storm_speed")

if RAINNC_VAR not in ds_other.data_vars or RAINC_VAR not in ds_other.data_vars:
    raise ValueError(f"'{RAINNC_VAR}'/'{RAINC_VAR}' not found in other_hazard_nc.")
da_rain = (ds_other[RAINNC_VAR] + ds_other[RAINC_VAR]).rename("rainfall")

# =========================
# Detect proper units
# =========================
speed_u = _clean_unit(ds_other[U10_VAR].attrs.get("units"), fallback="m/s")
speed_v = _clean_unit(ds_other[V10_VAR].attrs.get("units"), fallback=speed_u)
speed_unit = speed_u if speed_u == speed_v else f"{speed_u} (from {U10_VAR}), {speed_v} (from {V10_VAR})"

surge_unit = _clean_unit(ds_surge[SURGE_VAR].attrs.get("units"), fallback="m")

rainnc_unit = _clean_unit(ds_other[RAINNC_VAR].attrs.get("units"), fallback="mm")
rainc_unit = _clean_unit(ds_other[RAINC_VAR].attrs.get("units"), fallback=rainnc_unit)
rain_unit = rainnc_unit if rainnc_unit == rainc_unit else f"{rainnc_unit}+{rainc_unit}"

# =========================
# Zonal means
# =========================
pts_speed = _da_to_points(ds_other, da_speed, agg="max")
pts_surge = _da_to_points(ds_surge, da_surge, agg="max")
pts_rain = _da_to_points(ds_other, da_rain, agg="max")

storm_speed_vals = zonal_mean_from_points(pts_speed, gdf)
storm_surge_vals = zonal_mean_from_points(pts_surge, gdf)
rainfall_vals = zonal_mean_from_points(pts_rain, gdf)

# =========================
# Export (ONLY requested columns)
# =========================
df_out = pd.DataFrame({
    "Union": gdf[union_col].astype(str).values,
    f"Storm Speed ({speed_unit})": np.round(storm_speed_vals, 4),
    f"Storm Surge ({surge_unit})": np.round(storm_surge_vals, 4),
    f"Rainfall ({rain_unit})": np.round(rainfall_vals, 4),
})

df_out.to_csv(output_csv, index=False)
print(f"CSV exported: {output_csv}")
print(f"Detected units -> Speed: {speed_unit}, Surge: {surge_unit}, Rainfall: {rain_unit}")
print(df_out.head())

storm_surge_nc vars: ['z', 'uc', 'vc', 'psea', 'u', 'v', 'z0']
other_hazard_nc vars: ['u10', 'v10', 'rainnc', 'rainc']
CSV exported: output_union_damage.csv
Detected units -> Speed: m/s, Surge: m, Rainfall: mm
           Union  Storm Speed (m/s)  Storm Surge (m)  Rainfall (mm)
0         Amtali            17.1756           0.1987        61.6862
1    Arpangashia            16.7203           0.1882        62.9238
2  Atharogachhia            17.3406           0.0946        47.1082
3         Chaora            16.6457           0.1326        50.9775
4   ‌Gulisakhali            16.6457           0.0859        50.9775
